# Elon Musk Tweet Tracker (Jupyter Dashboard)

Run this notebook to view the live dashboard in Jupyter. 
It imports the logic from `elonmusk_tweet.py` and patches the display for notebook compatibility.

In [ ]:
"""
ELON TWEET COMPLETE (Refactored)
Combines Selenium Scraper (Source of Truth) with Financial Analytics.
Architecture: Modular Class-Based System
"""

import time
import requests
import json
import numpy as np
import sys
import threading
import re
import logging
import os
import argparse
from datetime import datetime, timezone, timedelta
from scipy.stats import poisson, nbinom
from typing import List, Dict, Optional, Any, Tuple

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
class Config:
    MANUAL_COUNT_FALLBACK = 468
    BASE_RATE = 55.0
    REFRESH_SECONDS = 300
    TRACKER_URL = "https://xtracker.polymarket.com/user/elonmusk"
    BANKROLL = 1000.0  # User Bankroll
    KELLY_FRACTION = 0.25 # Safety factor
    DISPERSION_PARAM = 0.1 # Alpha (Controls overdispersion: Var = Mean + Alpha*Mean^2)
    MARKETS_PAGE = "https://polymarket.com/pop-culture/tweets-markets"
    API_BASE_URL = "https://gamma-api.polymarket.com/events"
    API_HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    BRAVE_PATH = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
    LOG_FORMAT = '%(asctime)s - %(levelname)s - %(message)s'

# Setup Logging
logging.basicConfig(level=logging.INFO, format=Config.LOG_FORMAT)
logger = logging.getLogger("ElonTweet")

# WIN FIX
try:
    if hasattr(sys.stdout, 'reconfigure'):
        sys.stdout.reconfigure(encoding='utf-8')
except: pass

# ==========================================
# 🌐 POLYMARKET API
# ==========================================
class PolymarketAPI:
    @staticmethod
    def get_event_by_slug(slug: str) -> Optional[Dict]:
        """Fetch single event by slug"""
        try:
            params = {"slug": slug}
            resp = requests.get(Config.API_BASE_URL, params=params, headers=Config.API_HEADERS, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            if data and isinstance(data, list):
                return data[0]
            return None
        except Exception as e:
            logger.error(f"API Fetch Error (Slug: {slug}): {e}")
            return None

    @staticmethod
    def get_active_elon_events() -> List[Dict]:
        """
        Fetches active events related to 'Elon Musk' and 'Tweets' from Gamma API.
        Attempts specific query first, then broader query.
        """
        logger.info("Searching for active Elon Musk Tweet markets...")
        
        def fetch_and_filter(query):
            params = {"limit": 50, "closed": "false", "q": query}
            try:
                resp = requests.get(Config.API_BASE_URL, params=params, headers=Config.API_HEADERS, timeout=10)
                resp.raise_for_status()
                events = resp.json()
                valid = []
                for event in events:
                    title = event.get('title', '').lower()
                    # Flexible matching: Must have 'elon' AND ('tweet' OR 'count')
                    if 'elon' in title and ('tweet' in title or 'count' in title):
                         if event.get('closed') is False:
                            valid.append(event)
                return valid
            except Exception as e:
                logger.error(f"API Fetch Error ({query}): {e}")
                return []

        # 1. Try specific
        events = fetch_and_filter("Elon Musk Tweets")
        if events: 
            logger.info(f"Found {len(events)} events via specific query.")
            return events
            
        # 2. Try broad fallback
        logger.info("Specific query empty, trying broad 'Elon' search...")
        events = fetch_and_filter("Elon")
        logger.info(f"Found {len(events)} events via broad query.")
        return events

# ==========================================
# 🕵️ TRACKER (Selenium / Brave)
# ==========================================
class ElonTracker:
    def __init__(self, headless: bool = True):
        self.url = Config.TRACKER_URL
        self.driver = None
        self.last_data: Optional[List[Dict]] = None
        self.lock = threading.Lock()
        self.active = False
        self.headless = headless

    def __enter__(self):
        self.start()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

    def start(self):
        try:
            from selenium import webdriver
            from selenium.webdriver.chrome.options import Options
            from selenium.webdriver.chrome.service import Service
            
            options = Options()
            options.binary_location = Config.BRAVE_PATH
            if self.headless:
                options.add_argument("--headless=new")
            options.add_argument("--disable-gpu")
            options.add_argument("--log-level=3")
            options.add_argument("--no-first-run") 
            
            # Helper to find chromedriver if not in path? 
            # Usually selenium manager handles this now in recent versions.
            
            logger.info("🚀 Launching Tracker (Brave)...")
            self.driver = webdriver.Chrome(options=options)
            self.active = True
        except ImportError:
             logger.critical("Selenium not installed. Install with: pip install selenium")
             self.active = False
        except Exception as e:
            logger.error(f"❌ Browser Launch Error: {e}")
            self.active = False

    def scan_polymarket_page(self) -> List[str]:
        """Scans the configured markets page for event slugs."""
        if not self.active or not self.driver: return []
        slugs = []
        try:
            from selenium.webdriver.common.by import By
            from selenium.webdriver.support.ui import WebDriverWait
            from selenium.webdriver.support import expected_conditions as EC
            
            logger.info("🔎 Scanning Polymarket Page for new markets...")
            self.driver.get(Config.MARKETS_PAGE)
            WebDriverWait(self.driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "a")))
            time.sleep(3) # Allow hydration
            
            links = self.driver.find_elements(By.TAG_NAME, "a")
            for link in links:
                try:
                    href = link.get_attribute('href')
                    if href and '/event/' in href:
                        # Extract slug: https://polymarket.com/event/slug-text
                        parts = href.split('/event/')
                        if len(parts) > 1:
                            slug = parts[1].split('/')[0].split('?')[0]
                            slugs.append(slug)
                except: continue
                
            slugs = list(set(slugs))
            logger.info(f"   Found {len(slugs)} market slugs on page.")
            return slugs
        except Exception as e:
            logger.error(f"Error scanning markets page: {e}")
            return []

    def update(self):
        if not self.active or not self.driver: return
        try:
            from selenium.webdriver.common.by import By
            from selenium.webdriver.support.ui import WebDriverWait
            from selenium.webdriver.support import expected_conditions as EC
            
            logger.info("📡 Updating Counts from XTracker...")
            self.driver.get(self.url)
            WebDriverWait(self.driver, 25).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(5) # Allow dynamic content to load
            
            text = self.driver.find_element(By.TAG_NAME, "body").text
            self._parse_text(text)
            
        except Exception as e:
            logger.warning(f"⚠️ Scraping Warning: {e}")

    def _parse_text(self, text: str):
        date_pattern = re.compile(r"([A-Z][a-z]+ \d{1,2}(?:, \d{4})? - [A-Z][a-z]+ \d{1,2}(?:, \d{4})?)")
        found = []
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        
        for i, line in enumerate(lines):
            if date_pattern.search(line):
                # Look ahead for the number
                for j in range(1, 5):
                    if i+j >= len(lines): break
                    cand = lines[i+j].replace(',', '')
                    if cand.isdigit():
                        found.append({'range': line, 'count': int(cand)})
                        break
        
        if found:
            with self.lock:
                self.last_data = found
            logger.info(f"✅ Updated data: {len(found)} periods found.")
        else:
            logger.warning("No data patterns found in page text.")

    def get_data(self) -> Optional[List[Dict]]:
        with self.lock:
            return self.last_data
            
    def close(self):
        if self.driver: 
            try:
                self.driver.quit()
            except: pass
        self.active = False

# ==========================================
# 🧠 ANALYTICS
# ==========================================
class TweetAnalyzer:
    @staticmethod
    def get_texas_status() -> Tuple[float, str]:
        utc = datetime.now(timezone.utc)
        tx = utc - timedelta(hours=6)
        h = tx.hour
        if 3 <= h < 9: return 0.1, "💤 SLEEP"
        elif 22 <= h or h < 2: return 1.8, "🔥 MANIC"
        return 1.0, "🏢 WORK"

    @staticmethod
    def calculate_dynamic_rate(tracker_data: Optional[List[Dict]]) -> float:
        """
        Calculates average daily tweets from the most recent COMPLETED week.
        """
        if not tracker_data or len(tracker_data) < 2:
            return Config.BASE_RATE
            
        try:
            # Index 1 is typically the last full week
            last_full = tracker_data[1]
            count = last_full['count']
            return count / 7.0
        except:
            return Config.BASE_RATE

    @staticmethod
    def calculate_nbinom_prob(n_min: int, n_max: int, mu: float, days_left: float) -> float:
        """
        Calculates probability using Negative Binomial Distribution.
        Mu (Mean) = Projected Tweets
        Alpha (Dispersion) = Config.DISPERSION_PARAM
        
        DYNAMIC DISPERSION:
        As time remains decreases (days_left < 2), we reduce Alpha linearly to 0.
        This forces convergence to Poisson (Variance = Mean) as we approach the deadline,
        reducing "burstiness" assumption in the final hours.
        """
        if mu <= 0: return 0.0
        
        # Dynamic Alpha Calculation
        alpha = Config.DISPERSION_PARAM
        if days_left < 2.0:
            # Linear decay from 2.0 days down to 0
            decay = max(0.0, days_left / 2.0)
            alpha *= decay

        var = mu + alpha * (mu ** 2)
        
        # Scipy nbinom(n, p) parameterization:
        # p = mu / var
        # n = mu^2 / (var - mu)
        
        try:
            p = mu / var
            n = (mu ** 2) / (var - mu)
            
            # Probability mass in range [n_min, n_max]
            # CDF(Max) - CDF(Min - 1)
            prob = (nbinom.cdf(n_max, n, p) - nbinom.cdf(n_min - 1, n, p)) * 100
            return prob
        except:
             # Fallback to Poisson if something explodes (e.g. var <= mu which shouldn't happen with alpha > 0)
             return (poisson.cdf(n_max, mu) - poisson.cdf(n_min - 1, mu)) * 100

    @staticmethod
    def calculate_poisson_prob(n_min: int, n_max: int, mu: float) -> float:
        """
        Calculates probability using standard Poisson Distribution.
        Mean = Variance = Mu
        """
        if mu <= 0: return 0.0
        try:
            prob = (poisson.cdf(n_max, mu) - poisson.cdf(n_min - 1, mu)) * 100
            return prob
        except:
            return 0.0

    @staticmethod
    def calculate_kelly(prob_percent: float, price_cents: float, 
                       current_count: int, proj_count: int, days_left: float) -> Tuple[float, float, str]:
        """
        Calculates Kelly Criterion bet sizing with Adaptive Aggression.
        Returns: (fraction_of_bankroll, dollar_amount, reason)
        """
        if prob_percent <= 0 or price_cents <= 0 or price_cents >= 100:
            return 0.0, 0.0, "N/A"

        p = prob_percent / 100.0
        q = 1.0 - p
        b = (100.0 / price_cents) - 1.0 # Net odds received (decimal - 1)
        
        if b <= 0: return 0.0, 0.0, "NegOdds"

        # Kelly Formula: f* = (bp - q) / b
        f_star = (b * p - q) / b
        
        if f_star <= 0:
            return 0.0, 0.0, "NegEV"

        # Constraint: Max size < 1 / (DecimalOdds - 1) = 1/b
        # User request: "1/(max real wr)-1 posistion"
        # Interpreted as: Position Limit = 1 / (DecimalOdds - 1)
        # This ensures we don't risk more than the implied payout ratio allows?
        # Actually, 1/b is the logical upper bound for non-negative growth in some contexts.
        constraint = 1.0 / b
        
        # --- Standard Safety ---
        fraction = Config.KELLY_FRACTION
        
        # --- Adaptive Sensitivity ---
        # "Adjust the Kelly % calculation to be more aggressive when the Current Count 
        # is already within 10% of the Projected Count and time is running out."
        if days_left < 2.0 and proj_count > 0:
            diff = abs(current_count - proj_count)
            # If we are very close to projection (within 10%), increase confidence
            if diff <= 0.10 * proj_count:
                fraction = 0.5 # Boost to Half Kelly (Double the standard Quarter Kelly)
        
        safe_f = f_star * fraction
        
        # Apply User Constraint
        final_f = min(safe_f, constraint)
        
        amount = Config.BANKROLL * final_f
        return final_f, amount, "OK"

    @staticmethod
    def match_count(title: str, tracker_data: List[Dict]) -> Optional[int]:
        if not tracker_data: return None
        title_simp = title.lower().replace(" ", "").replace(",", "")
        
        # Try to find a partial match in the date range string
        for item in tracker_data:
            rng = item['range'].lower().replace(" ", "").replace(",", "")
            # Check if one is a substring of the other
            if title_simp in rng or rng in title_simp:
                return item['count']
        return None

# ==========================================
# 🖥️ DASHBOARD
# ==========================================
class Dashboard:
    @staticmethod
    def clear():
        # ANSI escape codes: \033[H (Home), \033[2J (Clear Screen)
        # This is more reliable in modern terminals/VS Code than os.system('cls')
        print("\033[H\033[2J", end="")
        sys.stdout.flush()

    @staticmethod
    def display(tracker_data: Optional[List[Dict]], events: List[Dict], api_mode: bool = False):
        Dashboard.clear()
        
        # 1. Header & Status
        dynamic_base = TweetAnalyzer.calculate_dynamic_rate(tracker_data)
        mult, status = TweetAnalyzer.get_texas_status()
        live_rate = dynamic_base * mult
        
        source = "🤖 AUTO (XTracker)" if tracker_data else f"🔴 MANUAL (Fallback: {Config.MANUAL_COUNT_FALLBACK})"
        print(f"{source}")
        print(f"🕵️ STATUS: {status} | ⚡ BASE: {dynamic_base:.1f} | 🔥 CLOCK: {live_rate:.1f}/day")
        print("─"*95)

        if not events:
            print("⚠️  NO ACTIVE MARKETS FOUND.")
            return

        utc_now = datetime.now(timezone.utc)
        
        # 2. Iterate Events
        for event in events:
            try:
                title = event['title']
                
                # Parse End Date
                end_str = event['endDate'].replace('Z', '+00:00')
                end = datetime.fromisoformat(end_str)
                days_left = (end - utc_now).total_seconds()/86400
                
                if days_left <= 0: continue

                # Get Count
                my_count = TweetAnalyzer.match_count(title, tracker_data) if tracker_data else None
                if my_count is None: my_count = Config.MANUAL_COUNT_FALLBACK
                
                # Projection
                impact = (live_rate - dynamic_base) * min(days_left, 0.2)
                proj = int(my_count + (dynamic_base * days_left) + impact)

                # Header
                print(f"\n📅 {title[:70]}")
                print(f"   🐦 Count: {my_count} | 🎯 Proj: {proj} | ⏳ Left: {days_left:.2f}d")
                print(f"   {'BUCKET':<12} {'PRICE':<8} {'PROB %':<8} {'EDGE':<8} {'EDGE(2)':<8} {'KELLY %':<8} {'SIZE ($)':<8} {'ACTION'}")
                print(f"   {'──────':<12} {'─────':<8} {'──────':<8} {'────':<8} {'───────':<8} {'───────':<8} {'────────':<8} {'──────'}")

                # Buckets
                markets = event.get('markets', [])
                buckets = Dashboard._parse_markets(markets)
                
                # --- ARBITRAGE CHECK ---
                total_price = sum(b['p'] for b in buckets)
                if total_price < 99.0: # Allow 1% buffer for safety/fees? or strict < 100?
                    # Cost = total_price, Payout = 100
                    # ROI = (100 - cost) / cost
                    roi = (100.0 - total_price) / total_price * 100.0
                    print(f"   🚨 ARBITRAGE OPPORTUNITY: Sum of Prices = {total_price:.1f}¢ (ROI: {roi:.2f}%) 🚨")
                    print(f"      ACTION: BUY ALL OUTCOMES")
                
                for b in buckets:
                    # Poisson logic
                    if my_count > b['h']: 
                        prob = 0.0
                        prob_pois = 0.0
                    else:
                        n_max = max(0, b['h'] - my_count)
                        n_min = max(0, b['l'] - my_count)
                        # nbinom logic via helper
                        remaining_proj = max(0, proj - my_count)
                        if remaining_proj == 0:
                             prob = 100.0 if (n_min == 0) else 0.0
                             prob_pois = prob
                        else:
                             # UPDATED: Pass days_left for dynamic dispersion
                             prob = TweetAnalyzer.calculate_nbinom_prob(n_min, n_max, remaining_proj, days_left)
                             prob_pois = TweetAnalyzer.calculate_poisson_prob(n_min, n_max, remaining_proj)
                    
                    edge = prob - b['p']
                    edge2 = prob_pois - b['p']
                    
                    # Kelly Calc (Updated with context)
                    kf, amt, k_reason = TweetAnalyzer.calculate_kelly(prob, b['p'], my_count, proj, days_left)
                    
                    # Signal formatting
                    sig, col = "-", "\033[0m"
                    if days_left > 0:
                        if my_count > b['h']: sig, col = "💀 DEAD", "\033[90m"
                        elif my_count >= b['l']:
                            if prob > 80: sig, col = "💎 HOLD", "\033[96m"
                            else: sig, col = "⚠️ WATCH", "\033[93m"
                        else:
                            if edge > 15: sig, col = "🚀 BUY YES", "\033[92m"
                            elif edge < -15: sig, col = "❌ BUY NO", "\033[91m"
                    
                    # Filter junk but keep valid lines
                    if b['p'] < 1.0 and prob < 1.0: continue 
                    
                    k_str = f"{kf*100:>4.1f}%" if kf > 0 else "-"
                    sz_str = f"${amt:>4.0f}" if amt > 0 else "-"
                    
                    print(f"   {b['n']:<12} {b['p']:>5.1f}¢   {prob:>5.1f}%    {col}{edge:+.1f}%   {edge2:+.1f}%\033[0m   {k_str:<8} {sz_str:<8} {sig}\033[0m")
                    
            except Exception as e:
                logger.error(f"Error processing event {event.get('title', 'Unknown')}: {e}")

    @staticmethod
    def _parse_markets(markets: List[Dict]) -> List[Dict]:
        buckets = []
        for m in markets:
            try:
                name = m.get('groupItemTitle', 'Unknown')
                l, h = 0, 9999
                
                # Parse Range
                if "-" in name: # "100-110"
                    p=name.split("-")
                    l, h = int(p[0]), int(p[1])
                elif "<" in name: # "<100"
                    h = int(name[1:]) - 1
                elif "+" in name: # "200+"
                    l = int(name[:-1])
                elif " or more" in name:
                     l = int(name.split(" ")[0])
                
                # Parse Price
                prices = json.loads(m.get('outcomePrices', '["0", "0"]'))
                price = float(prices[0]) * 100
                
                buckets.append({'l':l, 'h':h, 'p':price, 'n':name})
            except: 
                continue
        
        buckets.sort(key=lambda x: x['l'])
        return buckets

    @staticmethod
    def export_data(tracker_data, events, dynamic_base, live_rate, status):
         # Serialization logic similar to original, but robust
         pass

# ==========================================
# 🚀 MAIN LOOP
# ==========================================
def main():
    parser = argparse.ArgumentParser(description="Elon Tweet Tracker & Analyzer")
    parser.add_argument("--test", action="store_true", help="Run a single pass and exit")
    parser.add_argument("--no-browser", action="store_true", help="Disable browser tracking (Manual only)")
    
    # Handle Jupyter/Interactive environments
    try:
        if 'ipykernel_launcher' in sys.argv[0]:
            args = parser.parse_args([])
        else:
            args = parser.parse_args()
    except:
        # Fallback for other interactive modes
        args = parser.parse_args([])

    # Pass headless=False if debugging, or strictly True if user wants background
    # Since we are scanning a visual page, headless=True should still work for extraction
    tracker = ElonTracker(headless=True)
    
    try:
        if not args.no_browser:
            tracker.start()
        
        while True:
            # 1. Update Data
            if tracker.active:
                tracker.update()
                
            tracker_data = tracker.get_data()
            
            # 2. Scan Page Markets
            combined_events = []
            known_ids = set()
            
            # A. Search API
            search_events = PolymarketAPI.get_active_elon_events()
            for e in search_events:
                if e['id'] not in known_ids:
                    combined_events.append(e)
                    known_ids.add(e['id'])
                    
            # B. Page Scan
            if tracker.active:
                page_slugs = tracker.scan_polymarket_page()
                for slug in page_slugs:
                    # Avoid re-fetching if we already have it (unlikely unless searched)
                    # We don't have IDs for slugs yet, so just fetch
                    ev = PolymarketAPI.get_event_by_slug(slug)
                    if ev and ev['id'] not in known_ids:
                        # Double check it relates to Elon/Tweets? Or just trust the user?
                        # User said "scan for possible markets", implying all on that page.
                        # But let's be safe: filter closed
                        if ev.get('closed') is False:
                            combined_events.append(ev)
                            known_ids.add(ev['id'])
            
            # 3. Analytics & Display
            if not combined_events:
                 logger.info("No events found from Search or Page Scan.")
            
            Dashboard.display(tracker_data, combined_events)
            
            if args.test:
                print("\n✅ Test Pass Complete.")
                break
                
            # 4. Wait
            for i in range(Config.REFRESH_SECONDS, 0, -1):
                sys.stdout.write(f"\r💤 Refreshing in {i}s...")
                sys.stdout.flush()
                time.sleep(1)
                
    except KeyboardInterrupt:
        print("\n👋 Stopping...")
    finally:
        tracker.close()

if __name__ == "__main__":
    main()